# Data preparation for LoRA fine-tuning

This notebook loads `knowledge_base.json` (built by `python ingest.py`) and inspects the data used for supervised fine-tuning.

In [ ]:
import os
import sys
from pathlib import Path

# Project root (parent of notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

In [ ]:
import json
from pathlib import Path

import pandas as pd

from src.finetune_lib import build_dataset_rows, load_config

cfg = load_config(ROOT / "config.yaml")
data_cfg = cfg["data"]
kb_path = Path(data_cfg["processed_dir"]) / data_cfg["knowledge_base_file"]

with open(kb_path, encoding="utf-8") as f:
    kb = json.load(f)

docs = kb.get("documents", [])
print(f"Documents in KB: {len(docs)}")
print(f"KB file: {kb_path}")

In [ ]:
# Raw document fields (sample)
if docs:
    print(pd.json_normalize(docs[:3]).to_string())

In [ ]:
rows = build_dataset_rows(data_cfg["processed_dir"], data_cfg["knowledge_base_file"])
df = pd.DataFrame(rows)
print("Training rows (instruction/response):", len(df))
df.head()

In [ ]:
# Length distribution (characters)
df["instr_len"] = df["instruction"].str.len()
df["resp_len"] = df["response"].str.len()
df[["instr_len", "resp_len"]].describe()

In [ ]:
# Optional: preview one chat-formatted example (requires transformers tokenizer — quick or skip in 01)
from transformers import AutoTokenizer
ft = cfg.get("finetune", {})
mid = ft.get("base_model_id", "Qwen/Qwen2.5-3B-Instruct")
tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

from src.prompts import SYSTEM_PROMPT
r = rows[0]
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": r["instruction"]},
    {"role": "assistant", "content": r["response"]},
]
print(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)[:1200])